In [2]:
import os
import yaml
import pandas as pd
import matplotlib.pyplot as plt
import time
import seaborn as sns  # optional, just for color if you like
import matplotlib.pyplot as plt
from scipy.stats import spearmanr
from sklearn.linear_model import LinearRegression
import numpy as np
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
import statsmodels.formula.api as smf

In [3]:
base_dir = "/Users/seohyon/resources/results"

# will track only these methods (above BBKNN)
keep_methods = [
    "embed_cell_types_jittered",
    "embed_cell_types",
    "shuffle_integration",
    "shuffle_integration_by_cell_type",
    "shuffle_integration",
    "scanvi",
    "combat",
    "scvi",
    "harmony",
    "harmonypy",
    "batchelor_fastmnn",
    "liger",
    "pyliger",
    "scalex",
    "uce",
    "no_integration",
    "no_integration_batch",
    "scanorama"
]

yaml_path = "/Users/seohyon/resources/results/run_2025-10-11_13-28-50/score_uns.yaml"
rows = []

with open(yaml_path, "r") as f:
    data = yaml.safe_load(f)

for entry in data:
    method = entry.get("method_id", "").strip().lower()

    # keep only desired methods
    if method not in keep_methods:
        continue

    metrics = entry.get("metric_ids", [])
    values = entry.get("metric_values", [])

    for m_id, m_val in zip(metrics, values):
        rows.append({
            "dataset_id": entry.get("dataset_id"),
            "method_id": method,
            "metric_id": m_id,
            "metric_value": m_val
        })

In [4]:
df = pd.DataFrame(rows)
df = df.dropna(subset=["metric_value"]).copy() # drop NaN
# df = df[df["metric_id"] != "kbet"].copy() # drop kbet
df = df[~df["metric_id"].isin(["kbet", "hvg_overlap"])].copy() # drop unwanted metrics
df["dataset_id"] = df["dataset_id"].str.split("/").str[-1] # clean the dataset id

In [5]:
meta = pd.read_csv("/Users/seohyon/resources/Batch_Integration_Dataset - Tabellenblatt3.csv")

df = pd.merge(df, meta, left_on="dataset_id", right_on="dataset", how="inner").drop(columns=["dataset"])

control_type_map = {
    "embed_cell_types": "positive",
    "embed_cell_types_jittered": "positive",
    "shuffle_integration": "negative",
    "shuffle_integration_by_batch": "negative",
    "shuffle_integration_by_cell_type": "negative",
    "no_integration": "baseline",
    "no_integration_batch": "baseline",
}

df["control_type"] = df["method_id"].map(control_type_map).fillna("non-control")
df["control_group"] = df["control_type"].apply(
    lambda x: "non-control method" if x == "non-control" else "control method"
)

# mapping categorical to binary for plotting
df["nested_batches_num"] = df["nested_batches"].map({"No": 0, "Yes": 1})

bio_metrics = [
    'ari',
    'nmi',
    'graph_connectivity',
    'cell_cycle_conservation',
    'asw_label',
    'isolated_label_f1',
    'isolated_label_asw',
]

batch_metrics = [
    'pcr',
    'ilisi',
    'clisi',
    'asw_batch',
    'kbet_pg',
    'kbet_pg_label',
    'ari_batch',
    'nmi_batch',
]

def categorize_metric(m):
    if m in bio_metrics:
        return "bio"
    if m in batch_metrics:
        return "batch"
    return "baseline"

df["metric_type"] = df["metric_id"].apply(categorize_metric)

In [6]:
baseline = df[df["metric_type"] == "baseline"]
non_baseline = df[df["metric_type"] != "baseline"]

# For non-baseline, the metric_group is just metric_type ("bio" or "batch")
non_baseline = non_baseline.assign(metric_group=non_baseline["metric_type"])

# Duplicate baseline rows: one copy for "bio", one for "batch"
baseline_bio = baseline.copy()
baseline_bio["metric_group"] = "bio"

baseline_batch = baseline.copy()
baseline_batch["metric_group"] = "batch"

# Combine everything
df = pd.concat(
    [non_baseline, baseline_bio, baseline_batch],
    ignore_index=True
)

In [7]:
df["metric_id"].unique()

array(['ilisi', 'clisi', 'cell_cycle_conservation', 'asw_label', 'ari',
       'nmi', 'ari_batch', 'nmi_batch', 'kbet_pg', 'graph_connectivity',
       'isolated_label_asw', 'asw_batch', 'pcr', 'kbet_pg_label',
       'isolated_label_f1'], dtype=object)

In [8]:
df["dataset_id"].unique()

array(['gtex_v9', 'hypomap', 'mouse_pancreas_atlas', 'dkd',
       'immune_cell_atlas', 'tabula_sapiens'], dtype=object)

In [9]:
df["method_id"].unique()

array(['embed_cell_types', 'batchelor_fastmnn',
       'embed_cell_types_jittered', 'scanvi', 'liger', 'harmonypy',
       'scalex', 'combat', 'harmony', 'uce', 'shuffle_integration',
       'scvi', 'pyliger', 'no_integration', 'no_integration_batch',
       'scanorama', 'shuffle_integration_by_cell_type'], dtype=object)

# with kbet_pg

In [12]:
# This is good
model = smf.ols("metric_value ~ batch_imbalance_gc", data=df.query("metric_id == 'kbet_pg' and method_id == 'batchelor_fastmnn'")).fit()
print(model.summary())


                            OLS Regression Results                            
Dep. Variable:           metric_value   R-squared:                       0.707
Model:                            OLS   Adj. R-squared:                  0.634
Method:                 Least Squares   F-statistic:                     9.663
Date:                Mon, 08 Dec 2025   Prob (F-statistic):             0.0359
Time:                        18:17:25   Log-Likelihood:                 2.0679
No. Observations:                   6   AIC:                           -0.1359
Df Residuals:                       4   BIC:                           -0.5523
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                         coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------
Intercept             -0.5030      0

/Users/seohyon/miniconda3/envs/scanpy/lib/python3.10/site-packages/statsmodels/stats/stattools.py:74: ValueWarning: omni_normtest is not valid with less than 8 observations; 6 samples were given.
  warn("omni_normtest is not valid with less than 8 observations; %i "


In [13]:
model = smf.ols("metric_value ~ batch_imbalance_gc", data=df.query("metric_id == 'kbet_pg' and method_id == 'embed_cell_types'")).fit()
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:           metric_value   R-squared:                       0.127
Model:                            OLS   Adj. R-squared:                 -0.091
Method:                 Least Squares   F-statistic:                    0.5836
Date:                Mon, 08 Dec 2025   Prob (F-statistic):              0.487
Time:                        18:20:56   Log-Likelihood:               -0.27722
No. Observations:                   6   AIC:                             4.554
Df Residuals:                       4   BIC:                             4.138
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                         coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------
Intercept              0.0849      0

/Users/seohyon/miniconda3/envs/scanpy/lib/python3.10/site-packages/statsmodels/stats/stattools.py:74: ValueWarning: omni_normtest is not valid with less than 8 observations; 6 samples were given.
  warn("omni_normtest is not valid with less than 8 observations; %i "


In [14]:
model = smf.ols("metric_value ~ batch_imbalance_gc", data=df.query("metric_id == 'kbet_pg' and method_id == 'embed_cell_types_jittered'")).fit()
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:           metric_value   R-squared:                       0.136
Model:                            OLS   Adj. R-squared:                 -0.080
Method:                 Least Squares   F-statistic:                    0.6315
Date:                Mon, 08 Dec 2025   Prob (F-statistic):              0.471
Time:                        18:22:26   Log-Likelihood:               -0.13541
No. Observations:                   6   AIC:                             4.271
Df Residuals:                       4   BIC:                             3.854
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                         coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------
Intercept              0.0804      0

/Users/seohyon/miniconda3/envs/scanpy/lib/python3.10/site-packages/statsmodels/stats/stattools.py:74: ValueWarning: omni_normtest is not valid with less than 8 observations; 6 samples were given.
  warn("omni_normtest is not valid with less than 8 observations; %i "


In [15]:
# this is good
model = smf.ols("metric_value ~ batch_imbalance_gc", data=df.query("metric_id == 'kbet_pg' and method_id == 'scanvi'")).fit()
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:           metric_value   R-squared:                       0.667
Model:                            OLS   Adj. R-squared:                  0.583
Method:                 Least Squares   F-statistic:                     7.996
Date:                Mon, 08 Dec 2025   Prob (F-statistic):             0.0475
Time:                        18:23:25   Log-Likelihood:                 1.7738
No. Observations:                   6   AIC:                            0.4524
Df Residuals:                       4   BIC:                           0.03592
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                         coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------
Intercept             -0.4849      0

/Users/seohyon/miniconda3/envs/scanpy/lib/python3.10/site-packages/statsmodels/stats/stattools.py:74: ValueWarning: omni_normtest is not valid with less than 8 observations; 6 samples were given.
  warn("omni_normtest is not valid with less than 8 observations; %i "


In [16]:
model = smf.ols("metric_value ~ batch_imbalance_gc", data=df.query("metric_id == 'kbet_pg' and method_id == 'liger'")).fit()
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:           metric_value   R-squared:                       0.421
Model:                            OLS   Adj. R-squared:                  0.277
Method:                 Least Squares   F-statistic:                     2.914
Date:                Mon, 08 Dec 2025   Prob (F-statistic):              0.163
Time:                        18:23:29   Log-Likelihood:                 1.3673
No. Observations:                   6   AIC:                             1.265
Df Residuals:                       4   BIC:                            0.8490
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                         coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------
Intercept             -0.1041      0

/Users/seohyon/miniconda3/envs/scanpy/lib/python3.10/site-packages/statsmodels/stats/stattools.py:74: ValueWarning: omni_normtest is not valid with less than 8 observations; 6 samples were given.
  warn("omni_normtest is not valid with less than 8 observations; %i "


In [17]:
# this is good
model = smf.ols("metric_value ~ batch_imbalance_gc", data=df.query("metric_id == 'kbet_pg' and method_id == 'harmonypy'")).fit()
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:           metric_value   R-squared:                       0.659
Model:                            OLS   Adj. R-squared:                  0.574
Method:                 Least Squares   F-statistic:                     7.737
Date:                Mon, 08 Dec 2025   Prob (F-statistic):             0.0497
Time:                        18:24:37   Log-Likelihood:                 1.8557
No. Observations:                   6   AIC:                            0.2887
Df Residuals:                       4   BIC:                           -0.1278
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                         coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------
Intercept             -0.4584      0

/Users/seohyon/miniconda3/envs/scanpy/lib/python3.10/site-packages/statsmodels/stats/stattools.py:74: ValueWarning: omni_normtest is not valid with less than 8 observations; 6 samples were given.
  warn("omni_normtest is not valid with less than 8 observations; %i "


In [18]:
# this is good
model = smf.ols("metric_value ~ batch_imbalance_gc", data=df.query("metric_id == 'kbet_pg' and method_id == 'scalex'")).fit()
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:           metric_value   R-squared:                       0.667
Model:                            OLS   Adj. R-squared:                  0.583
Method:                 Least Squares   F-statistic:                     8.001
Date:                Mon, 08 Dec 2025   Prob (F-statistic):             0.0474
Time:                        18:24:42   Log-Likelihood:                 1.3904
No. Observations:                   6   AIC:                             1.219
Df Residuals:                       4   BIC:                            0.8027
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                         coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------
Intercept             -0.5180      0

/Users/seohyon/miniconda3/envs/scanpy/lib/python3.10/site-packages/statsmodels/stats/stattools.py:74: ValueWarning: omni_normtest is not valid with less than 8 observations; 6 samples were given.
  warn("omni_normtest is not valid with less than 8 observations; %i "


In [19]:
# this is good
model = smf.ols("metric_value ~ batch_imbalance_gc", data=df.query("metric_id == 'kbet_pg' and method_id == 'combat'")).fit()
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:           metric_value   R-squared:                       0.688
Model:                            OLS   Adj. R-squared:                  0.609
Method:                 Least Squares   F-statistic:                     8.801
Date:                Mon, 08 Dec 2025   Prob (F-statistic):             0.0413
Time:                        18:24:46   Log-Likelihood:                 1.4450
No. Observations:                   6   AIC:                             1.110
Df Residuals:                       4   BIC:                            0.6935
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                         coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------
Intercept             -0.5512      0

/Users/seohyon/miniconda3/envs/scanpy/lib/python3.10/site-packages/statsmodels/stats/stattools.py:74: ValueWarning: omni_normtest is not valid with less than 8 observations; 6 samples were given.
  warn("omni_normtest is not valid with less than 8 observations; %i "


In [20]:
# this is not good - p-vale is not smaller than 0.05
model = smf.ols("metric_value ~ batch_imbalance_gc", data=df.query("metric_id == 'kbet_pg' and method_id == 'harmony'")).fit()
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:           metric_value   R-squared:                       0.658
Model:                            OLS   Adj. R-squared:                  0.573
Method:                 Least Squares   F-statistic:                     7.710
Date:                Mon, 08 Dec 2025   Prob (F-statistic):             0.0500
Time:                        18:25:38   Log-Likelihood:                 1.8575
No. Observations:                   6   AIC:                            0.2849
Df Residuals:                       4   BIC:                           -0.1315
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                         coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------
Intercept             -0.4556      0

/Users/seohyon/miniconda3/envs/scanpy/lib/python3.10/site-packages/statsmodels/stats/stattools.py:74: ValueWarning: omni_normtest is not valid with less than 8 observations; 6 samples were given.
  warn("omni_normtest is not valid with less than 8 observations; %i "


In [21]:
# this is good
model = smf.ols("metric_value ~ batch_imbalance_gc", data=df.query("metric_id == 'kbet_pg' and method_id == 'uce'")).fit()
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:           metric_value   R-squared:                       0.711
Model:                            OLS   Adj. R-squared:                  0.639
Method:                 Least Squares   F-statistic:                     9.860
Date:                Mon, 08 Dec 2025   Prob (F-statistic):             0.0348
Time:                        18:25:41   Log-Likelihood:                 1.9655
No. Observations:                   6   AIC:                           0.06907
Df Residuals:                       4   BIC:                           -0.3474
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                         coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------
Intercept             -0.5293      0

/Users/seohyon/miniconda3/envs/scanpy/lib/python3.10/site-packages/statsmodels/stats/stattools.py:74: ValueWarning: omni_normtest is not valid with less than 8 observations; 6 samples were given.
  warn("omni_normtest is not valid with less than 8 observations; %i "


In [22]:
# this is good - and interesting (slope negative)
model = smf.ols("metric_value ~ batch_imbalance_gc", data=df.query("metric_id == 'kbet_pg' and method_id == 'shuffle_integration'")).fit()
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:           metric_value   R-squared:                       0.801
Model:                            OLS   Adj. R-squared:                  0.751
Method:                 Least Squares   F-statistic:                     16.09
Date:                Mon, 08 Dec 2025   Prob (F-statistic):             0.0160
Time:                        18:25:47   Log-Likelihood:                 15.016
No. Observations:                   6   AIC:                            -26.03
Df Residuals:                       4   BIC:                            -26.45
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                         coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------
Intercept              1.0039      0

/Users/seohyon/miniconda3/envs/scanpy/lib/python3.10/site-packages/statsmodels/stats/stattools.py:74: ValueWarning: omni_normtest is not valid with less than 8 observations; 6 samples were given.
  warn("omni_normtest is not valid with less than 8 observations; %i "


In [23]:
# this is good
model = smf.ols("metric_value ~ batch_imbalance_gc", data=df.query("metric_id == 'kbet_pg' and method_id == 'scvi'")).fit()
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:           metric_value   R-squared:                       0.681
Model:                            OLS   Adj. R-squared:                  0.602
Method:                 Least Squares   F-statistic:                     8.553
Date:                Mon, 08 Dec 2025   Prob (F-statistic):             0.0430
Time:                        18:25:51   Log-Likelihood:                 1.8231
No. Observations:                   6   AIC:                            0.3538
Df Residuals:                       4   BIC:                          -0.06263
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                         coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------
Intercept             -0.4974      0

/Users/seohyon/miniconda3/envs/scanpy/lib/python3.10/site-packages/statsmodels/stats/stattools.py:74: ValueWarning: omni_normtest is not valid with less than 8 observations; 6 samples were given.
  warn("omni_normtest is not valid with less than 8 observations; %i "


In [24]:
model = smf.ols("metric_value ~ batch_imbalance_gc", data=df.query("metric_id == 'kbet_pg' and method_id == 'pyliger'")).fit()
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:           metric_value   R-squared:                       0.390
Model:                            OLS   Adj. R-squared:                  0.237
Method:                 Least Squares   F-statistic:                     2.557
Date:                Mon, 08 Dec 2025   Prob (F-statistic):              0.185
Time:                        18:31:54   Log-Likelihood:                 1.1193
No. Observations:                   6   AIC:                             1.761
Df Residuals:                       4   BIC:                             1.345
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                         coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------
Intercept             -0.0892      0

/Users/seohyon/miniconda3/envs/scanpy/lib/python3.10/site-packages/statsmodels/stats/stattools.py:74: ValueWarning: omni_normtest is not valid with less than 8 observations; 6 samples were given.
  warn("omni_normtest is not valid with less than 8 observations; %i "


In [26]:
# this is good
model = smf.ols("metric_value ~ batch_imbalance_gc", data=df.query("metric_id == 'kbet_pg' and method_id == 'no_integration'")).fit()
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:           metric_value   R-squared:                       0.695
Model:                            OLS   Adj. R-squared:                  0.619
Method:                 Least Squares   F-statistic:                     9.129
Date:                Mon, 08 Dec 2025   Prob (F-statistic):             0.0391
Time:                        18:32:10   Log-Likelihood:                 1.8044
No. Observations:                   6   AIC:                            0.3912
Df Residuals:                       4   BIC:                          -0.02529
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                         coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------
Intercept             -0.5275      0

/Users/seohyon/miniconda3/envs/scanpy/lib/python3.10/site-packages/statsmodels/stats/stattools.py:74: ValueWarning: omni_normtest is not valid with less than 8 observations; 6 samples were given.
  warn("omni_normtest is not valid with less than 8 observations; %i "


In [27]:
# this is good
model = smf.ols("metric_value ~ batch_imbalance_gc", data=df.query("metric_id == 'kbet_pg' and method_id == 'no_integration_batch'")).fit()
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:           metric_value   R-squared:                       0.688
Model:                            OLS   Adj. R-squared:                  0.610
Method:                 Least Squares   F-statistic:                     8.820
Date:                Mon, 08 Dec 2025   Prob (F-statistic):             0.0412
Time:                        18:32:25   Log-Likelihood:                 1.3963
No. Observations:                   6   AIC:                             1.207
Df Residuals:                       4   BIC:                            0.7909
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                         coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------
Intercept             -0.5574      0

/Users/seohyon/miniconda3/envs/scanpy/lib/python3.10/site-packages/statsmodels/stats/stattools.py:74: ValueWarning: omni_normtest is not valid with less than 8 observations; 6 samples were given.
  warn("omni_normtest is not valid with less than 8 observations; %i "


In [30]:
model = smf.ols("metric_value ~ batch_imbalance_gc", data=df.query("metric_id == 'kbet_pg' and method_id == 'scanorama'")).fit()
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:           metric_value   R-squared:                       0.125
Model:                            OLS   Adj. R-squared:                 -0.094
Method:                 Least Squares   F-statistic:                    0.5698
Date:                Mon, 08 Dec 2025   Prob (F-statistic):              0.492
Time:                        18:33:14   Log-Likelihood:                -2.8510
No. Observations:                   6   AIC:                             9.702
Df Residuals:                       4   BIC:                             9.285
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                         coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------
Intercept             -0.0516      0

/Users/seohyon/miniconda3/envs/scanpy/lib/python3.10/site-packages/statsmodels/stats/stattools.py:74: ValueWarning: omni_normtest is not valid with less than 8 observations; 6 samples were given.
  warn("omni_normtest is not valid with less than 8 observations; %i "


In [29]:
model = smf.ols("metric_value ~ batch_imbalance_gc", data=df.query("metric_id == 'kbet_pg' and method_id == 'shuffle_integration_by_cell_type'")).fit()
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:           metric_value   R-squared:                       0.148
Model:                            OLS   Adj. R-squared:                 -0.065
Method:                 Least Squares   F-statistic:                    0.6943
Date:                Mon, 08 Dec 2025   Prob (F-statistic):              0.452
Time:                        18:32:47   Log-Likelihood:               0.037812
No. Observations:                   6   AIC:                             3.924
Df Residuals:                       4   BIC:                             3.508
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                         coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------
Intercept              0.1026      0

/Users/seohyon/miniconda3/envs/scanpy/lib/python3.10/site-packages/statsmodels/stats/stattools.py:74: ValueWarning: omni_normtest is not valid with less than 8 observations; 6 samples were given.
  warn("omni_normtest is not valid with less than 8 observations; %i "


# with kbet_pg_label

In [31]:
model = smf.ols("metric_value ~ batch_imbalance_gc", data=df.query("metric_id == 'kbet_pg_label' and method_id == 'embed_cell_types'")).fit()
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:           metric_value   R-squared:                       0.277
Model:                            OLS   Adj. R-squared:                  0.097
Method:                 Least Squares   F-statistic:                     1.535
Date:                Mon, 08 Dec 2025   Prob (F-statistic):              0.283
Time:                        18:40:52   Log-Likelihood:                 11.020
No. Observations:                   6   AIC:                            -18.04
Df Residuals:                       4   BIC:                            -18.46
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                         coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------
Intercept              0.9653      0

/Users/seohyon/miniconda3/envs/scanpy/lib/python3.10/site-packages/statsmodels/stats/stattools.py:74: ValueWarning: omni_normtest is not valid with less than 8 observations; 6 samples were given.
  warn("omni_normtest is not valid with less than 8 observations; %i "


In [32]:
model = smf.ols("metric_value ~ batch_imbalance_gc", data=df.query("metric_id == 'kbet_pg_label' and method_id == 'embed_cell_types_jittered'")).fit()
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:           metric_value   R-squared:                       0.294
Model:                            OLS   Adj. R-squared:                  0.118
Method:                 Least Squares   F-statistic:                     1.666
Date:                Mon, 08 Dec 2025   Prob (F-statistic):              0.266
Time:                        18:40:59   Log-Likelihood:                 11.128
No. Observations:                   6   AIC:                            -18.26
Df Residuals:                       4   BIC:                            -18.67
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                         coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------
Intercept              0.9669      0

/Users/seohyon/miniconda3/envs/scanpy/lib/python3.10/site-packages/statsmodels/stats/stattools.py:74: ValueWarning: omni_normtest is not valid with less than 8 observations; 6 samples were given.
  warn("omni_normtest is not valid with less than 8 observations; %i "


In [33]:
# this is good
model = smf.ols("metric_value ~ batch_imbalance_gc", data=df.query("metric_id == 'kbet_pg_label' and method_id == 'batchelor_fastmnn'")).fit()
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:           metric_value   R-squared:                       0.913
Model:                            OLS   Adj. R-squared:                  0.892
Method:                 Least Squares   F-statistic:                     42.11
Date:                Mon, 08 Dec 2025   Prob (F-statistic):            0.00291
Time:                        18:41:10   Log-Likelihood:                 8.5060
No. Observations:                   6   AIC:                            -13.01
Df Residuals:                       4   BIC:                            -13.43
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                         coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------
Intercept             -0.0325      0

/Users/seohyon/miniconda3/envs/scanpy/lib/python3.10/site-packages/statsmodels/stats/stattools.py:74: ValueWarning: omni_normtest is not valid with less than 8 observations; 6 samples were given.
  warn("omni_normtest is not valid with less than 8 observations; %i "


In [34]:
# this is good
model = smf.ols("metric_value ~ batch_imbalance_gc", data=df.query("metric_id == 'kbet_pg_label' and method_id == 'scanvi'")).fit()
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:           metric_value   R-squared:                       0.893
Model:                            OLS   Adj. R-squared:                  0.866
Method:                 Least Squares   F-statistic:                     33.22
Date:                Mon, 08 Dec 2025   Prob (F-statistic):            0.00450
Time:                        18:41:16   Log-Likelihood:                 6.9453
No. Observations:                   6   AIC:                            -9.891
Df Residuals:                       4   BIC:                            -10.31
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                         coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------
Intercept             -0.1586      0

/Users/seohyon/miniconda3/envs/scanpy/lib/python3.10/site-packages/statsmodels/stats/stattools.py:74: ValueWarning: omni_normtest is not valid with less than 8 observations; 6 samples were given.
  warn("omni_normtest is not valid with less than 8 observations; %i "


In [35]:
model = smf.ols("metric_value ~ batch_imbalance_gc", data=df.query("metric_id == 'kbet_pg_label' and method_id == 'liger'")).fit()
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:           metric_value   R-squared:                       0.280
Model:                            OLS   Adj. R-squared:                  0.100
Method:                 Least Squares   F-statistic:                     1.554
Date:                Mon, 08 Dec 2025   Prob (F-statistic):              0.281
Time:                        18:41:22   Log-Likelihood:                 3.1536
No. Observations:                   6   AIC:                            -2.307
Df Residuals:                       4   BIC:                            -2.724
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                         coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------
Intercept              0.3750      0

/Users/seohyon/miniconda3/envs/scanpy/lib/python3.10/site-packages/statsmodels/stats/stattools.py:74: ValueWarning: omni_normtest is not valid with less than 8 observations; 6 samples were given.
  warn("omni_normtest is not valid with less than 8 observations; %i "


In [36]:
model = smf.ols("metric_value ~ batch_imbalance_gc", data=df.query("metric_id == 'kbet_pg_label' and method_id == 'pyliger'")).fit()
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:           metric_value   R-squared:                       0.212
Model:                            OLS   Adj. R-squared:                  0.014
Method:                 Least Squares   F-statistic:                     1.073
Date:                Mon, 08 Dec 2025   Prob (F-statistic):              0.359
Time:                        18:41:26   Log-Likelihood:                 2.5574
No. Observations:                   6   AIC:                            -1.115
Df Residuals:                       4   BIC:                            -1.531
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                         coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------
Intercept              0.3952      0

/Users/seohyon/miniconda3/envs/scanpy/lib/python3.10/site-packages/statsmodels/stats/stattools.py:74: ValueWarning: omni_normtest is not valid with less than 8 observations; 6 samples were given.
  warn("omni_normtest is not valid with less than 8 observations; %i "


In [37]:
# this is good
model = smf.ols("metric_value ~ batch_imbalance_gc", data=df.query("metric_id == 'kbet_pg_label' and method_id == 'harmony'")).fit()
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:           metric_value   R-squared:                       0.798
Model:                            OLS   Adj. R-squared:                  0.748
Method:                 Least Squares   F-statistic:                     15.84
Date:                Mon, 08 Dec 2025   Prob (F-statistic):             0.0164
Time:                        18:41:32   Log-Likelihood:                 5.8878
No. Observations:                   6   AIC:                            -7.776
Df Residuals:                       4   BIC:                            -8.192
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                         coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------
Intercept             -0.0486      0

/Users/seohyon/miniconda3/envs/scanpy/lib/python3.10/site-packages/statsmodels/stats/stattools.py:74: ValueWarning: omni_normtest is not valid with less than 8 observations; 6 samples were given.
  warn("omni_normtest is not valid with less than 8 observations; %i "


In [38]:
# this is good
model = smf.ols("metric_value ~ batch_imbalance_gc", data=df.query("metric_id == 'kbet_pg_label' and method_id == 'harmonypy'")).fit()
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:           metric_value   R-squared:                       0.797
Model:                            OLS   Adj. R-squared:                  0.746
Method:                 Least Squares   F-statistic:                     15.67
Date:                Mon, 08 Dec 2025   Prob (F-statistic):             0.0167
Time:                        18:41:40   Log-Likelihood:                 5.9743
No. Observations:                   6   AIC:                            -7.949
Df Residuals:                       4   BIC:                            -8.365
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                         coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------
Intercept             -0.0376      0

/Users/seohyon/miniconda3/envs/scanpy/lib/python3.10/site-packages/statsmodels/stats/stattools.py:74: ValueWarning: omni_normtest is not valid with less than 8 observations; 6 samples were given.
  warn("omni_normtest is not valid with less than 8 observations; %i "


In [39]:
# this is good
model = smf.ols("metric_value ~ batch_imbalance_gc", data=df.query("metric_id == 'kbet_pg_label' and method_id == 'scalex'")).fit()
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:           metric_value   R-squared:                       0.827
Model:                            OLS   Adj. R-squared:                  0.784
Method:                 Least Squares   F-statistic:                     19.12
Date:                Mon, 08 Dec 2025   Prob (F-statistic):             0.0119
Time:                        18:41:48   Log-Likelihood:                 4.5781
No. Observations:                   6   AIC:                            -5.156
Df Residuals:                       4   BIC:                            -5.573
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                         coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------
Intercept             -0.2522      0

/Users/seohyon/miniconda3/envs/scanpy/lib/python3.10/site-packages/statsmodels/stats/stattools.py:74: ValueWarning: omni_normtest is not valid with less than 8 observations; 6 samples were given.
  warn("omni_normtest is not valid with less than 8 observations; %i "


In [40]:
# this is good
model = smf.ols("metric_value ~ batch_imbalance_gc", data=df.query("metric_id == 'kbet_pg_label' and method_id == 'combat'")).fit()
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:           metric_value   R-squared:                       0.906
Model:                            OLS   Adj. R-squared:                  0.882
Method:                 Least Squares   F-statistic:                     38.40
Date:                Mon, 08 Dec 2025   Prob (F-statistic):            0.00345
Time:                        18:41:53   Log-Likelihood:                 5.9099
No. Observations:                   6   AIC:                            -7.820
Df Residuals:                       4   BIC:                            -8.236
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                         coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------
Intercept             -0.3831      0

/Users/seohyon/miniconda3/envs/scanpy/lib/python3.10/site-packages/statsmodels/stats/stattools.py:74: ValueWarning: omni_normtest is not valid with less than 8 observations; 6 samples were given.
  warn("omni_normtest is not valid with less than 8 observations; %i "


In [41]:
# this is good
model = smf.ols("metric_value ~ batch_imbalance_gc", data=df.query("metric_id == 'kbet_pg_label' and method_id == 'uce'")).fit()
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:           metric_value   R-squared:                       0.958
Model:                            OLS   Adj. R-squared:                  0.948
Method:                 Least Squares   F-statistic:                     92.28
Date:                Mon, 08 Dec 2025   Prob (F-statistic):           0.000656
Time:                        18:42:00   Log-Likelihood:                 9.0016
No. Observations:                   6   AIC:                            -14.00
Df Residuals:                       4   BIC:                            -14.42
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                         coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------
Intercept             -0.2617      0

/Users/seohyon/miniconda3/envs/scanpy/lib/python3.10/site-packages/statsmodels/stats/stattools.py:74: ValueWarning: omni_normtest is not valid with less than 8 observations; 6 samples were given.
  warn("omni_normtest is not valid with less than 8 observations; %i "


In [42]:
model = smf.ols("metric_value ~ batch_imbalance_gc", data=df.query("metric_id == 'kbet_pg_label' and method_id == 'shuffle_integration'")).fit()
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:           metric_value   R-squared:                       0.240
Model:                            OLS   Adj. R-squared:                  0.051
Method:                 Least Squares   F-statistic:                     1.266
Date:                Mon, 08 Dec 2025   Prob (F-statistic):              0.323
Time:                        18:42:06   Log-Likelihood:                 11.269
No. Observations:                   6   AIC:                            -18.54
Df Residuals:                       4   BIC:                            -18.95
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                         coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------
Intercept              0.9559      0

/Users/seohyon/miniconda3/envs/scanpy/lib/python3.10/site-packages/statsmodels/stats/stattools.py:74: ValueWarning: omni_normtest is not valid with less than 8 observations; 6 samples were given.
  warn("omni_normtest is not valid with less than 8 observations; %i "


In [43]:
model = smf.ols("metric_value ~ batch_imbalance_gc", data=df.query("metric_id == 'kbet_pg_label' and method_id == 'shuffle_integration_by_cell_type'")).fit()
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:           metric_value   R-squared:                       0.183
Model:                            OLS   Adj. R-squared:                 -0.022
Method:                 Least Squares   F-statistic:                    0.8943
Date:                Mon, 08 Dec 2025   Prob (F-statistic):              0.398
Time:                        18:42:13   Log-Likelihood:                 11.022
No. Observations:                   6   AIC:                            -18.04
Df Residuals:                       4   BIC:                            -18.46
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                         coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------
Intercept              0.9511      0

/Users/seohyon/miniconda3/envs/scanpy/lib/python3.10/site-packages/statsmodels/stats/stattools.py:74: ValueWarning: omni_normtest is not valid with less than 8 observations; 6 samples were given.
  warn("omni_normtest is not valid with less than 8 observations; %i "


In [44]:
# this is good
model = smf.ols("metric_value ~ batch_imbalance_gc", data=df.query("metric_id == 'kbet_pg_label' and method_id == 'no_integration'")).fit()
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:           metric_value   R-squared:                       0.931
Model:                            OLS   Adj. R-squared:                  0.914
Method:                 Least Squares   F-statistic:                     54.29
Date:                Mon, 08 Dec 2025   Prob (F-statistic):            0.00181
Time:                        18:42:19   Log-Likelihood:                 7.2166
No. Observations:                   6   AIC:                            -10.43
Df Residuals:                       4   BIC:                            -10.85
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                         coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------
Intercept             -0.3421      0

/Users/seohyon/miniconda3/envs/scanpy/lib/python3.10/site-packages/statsmodels/stats/stattools.py:74: ValueWarning: omni_normtest is not valid with less than 8 observations; 6 samples were given.
  warn("omni_normtest is not valid with less than 8 observations; %i "


In [46]:
# this is good
model = smf.ols("metric_value ~ batch_imbalance_gc", data=df.query("metric_id == 'kbet_pg_label' and method_id == 'no_integration_batch'")).fit()
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:           metric_value   R-squared:                       0.920
Model:                            OLS   Adj. R-squared:                  0.900
Method:                 Least Squares   F-statistic:                     46.03
Date:                Mon, 08 Dec 2025   Prob (F-statistic):            0.00246
Time:                        18:42:35   Log-Likelihood:                 5.8530
No. Observations:                   6   AIC:                            -7.706
Df Residuals:                       4   BIC:                            -8.123
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                         coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------
Intercept             -0.5013      0

/Users/seohyon/miniconda3/envs/scanpy/lib/python3.10/site-packages/statsmodels/stats/stattools.py:74: ValueWarning: omni_normtest is not valid with less than 8 observations; 6 samples were given.
  warn("omni_normtest is not valid with less than 8 observations; %i "


In [47]:
model = smf.ols("metric_value ~ batch_imbalance_gc", data=df.query("metric_id == 'kbet_pg_label' and method_id == 'scanorama'")).fit()
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:           metric_value   R-squared:                       0.305
Model:                            OLS   Adj. R-squared:                  0.131
Method:                 Least Squares   F-statistic:                     1.757
Date:                Mon, 08 Dec 2025   Prob (F-statistic):              0.256
Time:                        18:42:44   Log-Likelihood:               -0.31927
No. Observations:                   6   AIC:                             4.639
Df Residuals:                       4   BIC:                             4.222
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                         coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------
Intercept              0.0597      0

/Users/seohyon/miniconda3/envs/scanpy/lib/python3.10/site-packages/statsmodels/stats/stattools.py:74: ValueWarning: omni_normtest is not valid with less than 8 observations; 6 samples were given.
  warn("omni_normtest is not valid with less than 8 observations; %i "


In [48]:
# this is good
model = smf.ols("metric_value ~ batch_imbalance_gc", data=df.query("metric_id == 'kbet_pg_label' and method_id == 'scvi'")).fit()
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:           metric_value   R-squared:                       0.900
Model:                            OLS   Adj. R-squared:                  0.875
Method:                 Least Squares   F-statistic:                     35.91
Date:                Mon, 08 Dec 2025   Prob (F-statistic):            0.00390
Time:                        18:42:48   Log-Likelihood:                 6.8084
No. Observations:                   6   AIC:                            -9.617
Df Residuals:                       4   BIC:                            -10.03
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                         coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------
Intercept             -0.2083      0

/Users/seohyon/miniconda3/envs/scanpy/lib/python3.10/site-packages/statsmodels/stats/stattools.py:74: ValueWarning: omni_normtest is not valid with less than 8 observations; 6 samples were given.
  warn("omni_normtest is not valid with less than 8 observations; %i "


# with ilisi

- Threshold: p < 0.05 && R^2 > 0.06

In [11]:
rows = []

for method, df_m in df.query("metric_id == 'ilisi' and control_group == 'non-control method'").groupby("method_id"):
    # skip methods with too few points or no variation
    if df_m["batch_imbalance_gc"].nunique() < 2 or len(df_m) < 4:
        continue

    res = smf.ols("metric_value ~ batch_imbalance_gc", data=df_m).fit()

    rows.append({
        "method_id": method,
        "n": len(df_m),
        "slope": res.params["batch_imbalance_gc"],
        "pval": res.pvalues["batch_imbalance_gc"],
        "r2": res.rsquared,
    })

results = pd.DataFrame(rows)

hits = results.query("r2 > 0.6 and pval < 0.05 and n >= 5")
print(hits.sort_values("r2", ascending=False))



   method_id  n     slope      pval        r2
7  scanorama  6 -0.406617  0.011036  0.833483
4      liger  6 -0.533873  0.040788  0.689275
5    pyliger  6 -0.533609  0.045059  0.674494


In [12]:
rows = []

for method, df_m in df.query("metric_id == 'ilisi' and control_group == 'control method'").groupby("method_id"):
    # skip methods with too few points or no variation
    if df_m["batch_imbalance_gc"].nunique() < 2 or len(df_m) < 4:
        continue

    res = smf.ols("metric_value ~ batch_imbalance_gc", data=df_m).fit()

    rows.append({
        "method_id": method,
        "n": len(df_m),
        "slope": res.params["batch_imbalance_gc"],
        "pval": res.pvalues["batch_imbalance_gc"],
        "r2": res.rsquared,
    })

results = pd.DataFrame(rows)

# hits = results.query("r2 > 0.6 and pval < 0.05 and n >= 5")
print(results.sort_values("r2", ascending=False))


                          method_id  n     slope      pval        r2
4               shuffle_integration  6 -0.613100  0.034579  0.712441
1         embed_cell_types_jittered  6 -0.544293  0.049209  0.660888
0                  embed_cell_types  6 -0.550109  0.050148  0.657903
5  shuffle_integration_by_cell_type  6 -0.550909  0.051805  0.652714
2                    no_integration  6 -0.093173  0.064488  0.615900
3              no_integration_batch  6 -0.007925  0.133235  0.469181


- interesting - all negative slopes

# with clisi

In [84]:
rows = []

for method, df_m in df.query("metric_id == 'clisi'").groupby("method_id"):
    # skip methods with too few points or no variation
    if df_m["batch_imbalance_gc"].nunique() < 2 or len(df_m) < 4:
        continue

    res = smf.ols("metric_value ~ batch_imbalance_gc", data=df_m).fit()

    rows.append({
        "method_id": method,
        "n": len(df_m),
        "slope": res.params["batch_imbalance_gc"],
        "pval": res.pvalues["batch_imbalance_gc"],
        "r2": res.rsquared,
    })

results = pd.DataFrame(rows)

print(results.shape)
print(results.head())

hits = results.query("r2 > 0.6 and pval < 0.05 and n >= 5")
print(hits.sort_values("r2", ascending=False))

(17, 5)
                   method_id  n         slope      pval        r2
0          batchelor_fastmnn  6 -5.962260e-05  0.989150  0.000052
1                     combat  6  1.240721e-03  0.783659  0.021097
2           embed_cell_types  6 -1.110223e-16  0.844902      -inf
3  embed_cell_types_jittered  6  5.273559e-16  0.410726      -inf
4                    harmony  6  9.253072e-04  0.886461  0.005751
Empty DataFrame
Columns: [method_id, n, slope, pval, r2]
Index: []


/Users/seohyon/miniconda3/envs/scanpy/lib/python3.10/site-packages/statsmodels/regression/linear_model.py:1782: RuntimeWarning: divide by zero encountered in scalar divide
  return 1 - self.ssr/self.centered_tss
/Users/seohyon/miniconda3/envs/scanpy/lib/python3.10/site-packages/statsmodels/regression/linear_model.py:1782: RuntimeWarning: divide by zero encountered in scalar divide
  return 1 - self.ssr/self.centered_tss


- none of the fittings passed the threshold

# with cell_cycle_conservation

In [86]:
rows = []

for method, df_m in df.query("metric_id == 'cell_cycle_conservation'").groupby("method_id"):
    # skip methods with too few points or no variation
    if df_m["batch_imbalance_gc"].nunique() < 2 or len(df_m) < 4:
        continue

    res = smf.ols("metric_value ~ batch_imbalance_gc", data=df_m).fit()

    rows.append({
        "method_id": method,
        "n": len(df_m),
        "slope": res.params["batch_imbalance_gc"],
        "pval": res.pvalues["batch_imbalance_gc"],
        "r2": res.rsquared,
    })

results = pd.DataFrame(rows)

hits = results.query("r2 > 0.6 and pval < 0.05 and n >= 5")
print(hits.sort_values("r2", ascending=False))

        method_id  n     slope      pval        r2
4         harmony  6 -0.320734  0.005761  0.878660
5       harmonypy  6 -0.317122  0.006277  0.873470
7  no_integration  6 -0.348119  0.009990  0.841334


- all negative

# with asw_label

In [88]:
rows = []

for method, df_m in df.query("metric_id == 'asw_label'").groupby("method_id"):
    # skip methods with too few points or no variation
    if df_m["batch_imbalance_gc"].nunique() < 2 or len(df_m) < 4:
        continue

    res = smf.ols("metric_value ~ batch_imbalance_gc", data=df_m).fit()

    rows.append({
        "method_id": method,
        "n": len(df_m),
        "slope": res.params["batch_imbalance_gc"],
        "pval": res.pvalues["batch_imbalance_gc"],
        "r2": res.rsquared,
    })

results = pd.DataFrame(rows)

hits = results.query("r2 > 0.6 and pval < 0.05 and n >= 5")
print(hits.sort_values("r2", ascending=False))

Empty DataFrame
Columns: [method_id, n, slope, pval, r2]
Index: []


# with asw_batch

In [91]:
rows = []

for method, df_m in df.query("metric_id == 'asw_batch'").groupby("method_id"):
    # skip methods with too few points or no variation
    if df_m["batch_imbalance_gc"].nunique() < 2 or len(df_m) < 4:
        continue

    res = smf.ols("metric_value ~ batch_imbalance_gc", data=df_m).fit()

    rows.append({
        "method_id": method,
        "n": len(df_m),
        "slope": res.params["batch_imbalance_gc"],
        "pval": res.pvalues["batch_imbalance_gc"],
        "r2": res.rsquared,
    })

results = pd.DataFrame(rows)

hits = results.query("r2 > 0.6 and pval < 0.05 and n >= 5")
print(hits.sort_values("r2", ascending=False))

Empty DataFrame
Columns: [method_id, n, slope, pval, r2]
Index: []


# with ari

In [92]:
rows = []

for method, df_m in df.query("metric_id == 'ari'").groupby("method_id"):
    # skip methods with too few points or no variation
    if df_m["batch_imbalance_gc"].nunique() < 2 or len(df_m) < 4:
        continue

    res = smf.ols("metric_value ~ batch_imbalance_gc", data=df_m).fit()

    rows.append({
        "method_id": method,
        "n": len(df_m),
        "slope": res.params["batch_imbalance_gc"],
        "pval": res.pvalues["batch_imbalance_gc"],
        "r2": res.rsquared,
    })

results = pd.DataFrame(rows)

hits = results.query("r2 > 0.6 and pval < 0.05 and n >= 5")
print(hits.sort_values("r2", ascending=False))

            method_id  n     slope      pval        r2
6               liger  6 -0.774990  0.000952  0.950039
9             pyliger  6 -0.597343  0.003827  0.900701
13               scvi  6 -0.946169  0.004349  0.894276
4             harmony  6 -0.736895  0.008089  0.856807
1              combat  6 -0.731067  0.014597  0.809387
12             scanvi  6 -1.021971  0.015234  0.805423
0   batchelor_fastmnn  6 -0.492509  0.026973  0.744296


# with ari_batch

In [93]:
rows = []

for method, df_m in df.query("metric_id == 'ari_batch'").groupby("method_id"):
    # skip methods with too few points or no variation
    if df_m["batch_imbalance_gc"].nunique() < 2 or len(df_m) < 4:
        continue

    res = smf.ols("metric_value ~ batch_imbalance_gc", data=df_m).fit()

    rows.append({
        "method_id": method,
        "n": len(df_m),
        "slope": res.params["batch_imbalance_gc"],
        "pval": res.pvalues["batch_imbalance_gc"],
        "r2": res.rsquared,
    })

results = pd.DataFrame(rows)

hits = results.query("r2 > 0.6 and pval < 0.05 and n >= 5")
print(hits.sort_values("r2", ascending=False))

               method_id  n     slope      pval        r2
8   no_integration_batch  6  0.369722  0.006375  0.872501
16                   uce  6  0.172815  0.027938  0.739995


# with nmi

In [94]:
rows = []

for method, df_m in df.query("metric_id == 'nmi'").groupby("method_id"):
    # skip methods with too few points or no variation
    if df_m["batch_imbalance_gc"].nunique() < 2 or len(df_m) < 4:
        continue

    res = smf.ols("metric_value ~ batch_imbalance_gc", data=df_m).fit()

    rows.append({
        "method_id": method,
        "n": len(df_m),
        "slope": res.params["batch_imbalance_gc"],
        "pval": res.pvalues["batch_imbalance_gc"],
        "r2": res.rsquared,
    })

results = pd.DataFrame(rows)

hits = results.query("r2 > 0.6 and pval < 0.05 and n >= 5")
print(hits.sort_values("r2", ascending=False))

            method_id  n     slope      pval        r2
0   batchelor_fastmnn  6 -0.210902  0.005022  0.886545
4             harmony  6 -0.238947  0.006214  0.874085
13               scvi  6 -0.366778  0.010526  0.837258
1              combat  6 -0.294982  0.022654  0.764668
12             scanvi  6 -0.438074  0.023069  0.762623
5           harmonypy  6 -0.180513  0.036375  0.705513


# with nmi_batch

In [95]:
rows = []

for method, df_m in df.query("metric_id == 'nmi_batch'").groupby("method_id"):
    # skip methods with too few points or no variation
    if df_m["batch_imbalance_gc"].nunique() < 2 or len(df_m) < 4:
        continue

    res = smf.ols("metric_value ~ batch_imbalance_gc", data=df_m).fit()

    rows.append({
        "method_id": method,
        "n": len(df_m),
        "slope": res.params["batch_imbalance_gc"],
        "pval": res.pvalues["batch_imbalance_gc"],
        "r2": res.rsquared,
    })

results = pd.DataFrame(rows)

hits = results.query("r2 > 0.6 and pval < 0.05 and n >= 5")
print(hits.sort_values("r2", ascending=False))

Empty DataFrame
Columns: [method_id, n, slope, pval, r2]
Index: []


# with graph_connectivity

In [97]:
rows = []

for method, df_m in df.query("metric_id == 'graph_connectivity'").groupby("method_id"):
    # skip methods with too few points or no variation
    if df_m["batch_imbalance_gc"].nunique() < 2 or len(df_m) < 4:
        continue

    res = smf.ols("metric_value ~ batch_imbalance_gc", data=df_m).fit()

    rows.append({
        "method_id": method,
        "n": len(df_m),
        "slope": res.params["batch_imbalance_gc"],
        "pval": res.pvalues["batch_imbalance_gc"],
        "r2": res.rsquared,
    })

results = pd.DataFrame(rows)

hits = results.query("r2 > 0.6 and pval < 0.05 and n >= 5")
print(hits.sort_values("r2", ascending=False))

               method_id  n     slope      pval        r2
12                scanvi  6 -0.142713  0.012824  0.820936
8   no_integration_batch  6  0.477807  0.020240  0.776998
13                  scvi  6 -0.155221  0.035514  0.708811
16                   uce  6 -0.124816  0.046245  0.670534


# with isolated_label_asw

In [98]:
rows = []

for method, df_m in df.query("metric_id == 'isolated_label_asw'").groupby("method_id"):
    # skip methods with too few points or no variation
    if df_m["batch_imbalance_gc"].nunique() < 2 or len(df_m) < 4:
        continue

    res = smf.ols("metric_value ~ batch_imbalance_gc", data=df_m).fit()

    rows.append({
        "method_id": method,
        "n": len(df_m),
        "slope": res.params["batch_imbalance_gc"],
        "pval": res.pvalues["batch_imbalance_gc"],
        "r2": res.rsquared,
    })

results = pd.DataFrame(rows)

hits = results.query("r2 > 0.6 and pval < 0.05 and n >= 5")
print(hits.sort_values("r2", ascending=False))

                           method_id  n     slope      pval        r2
5                          harmonypy  5 -0.678306  0.003658  0.958315
4                            harmony  5 -0.690683  0.004809  0.950065
1                             combat  5 -0.616368  0.007091  0.935505
10                            scalex  5 -0.716049  0.012786  0.905070
15  shuffle_integration_by_cell_type  5 -0.735304  0.013035  0.903867
7                     no_integration  5 -0.735304  0.013035  0.903866
13                              scvi  5 -0.490589  0.014711  0.895971
12                            scanvi  5 -0.712846  0.015929  0.890441
0                  batchelor_fastmnn  5 -0.814218  0.021140  0.868332


# with isolated_label_f1

In [99]:
rows = []

for method, df_m in df.query("metric_id == 'isolated_label_f1'").groupby("method_id"):
    # skip methods with too few points or no variation
    if df_m["batch_imbalance_gc"].nunique() < 2 or len(df_m) < 4:
        continue

    res = smf.ols("metric_value ~ batch_imbalance_gc", data=df_m).fit()

    rows.append({
        "method_id": method,
        "n": len(df_m),
        "slope": res.params["batch_imbalance_gc"],
        "pval": res.pvalues["batch_imbalance_gc"],
        "r2": res.rsquared,
    })

results = pd.DataFrame(rows)

hits = results.query("r2 > 0.6 and pval < 0.05 and n >= 5")
print(hits.sort_values("r2", ascending=False))

Empty DataFrame
Columns: [method_id, n, slope, pval, r2]
Index: []


# with pcr

In [100]:
rows = []

for method, df_m in df.query("metric_id == 'pcr'").groupby("method_id"):
    # skip methods with too few points or no variation
    if df_m["batch_imbalance_gc"].nunique() < 2 or len(df_m) < 4:
        continue

    res = smf.ols("metric_value ~ batch_imbalance_gc", data=df_m).fit()

    rows.append({
        "method_id": method,
        "n": len(df_m),
        "slope": res.params["batch_imbalance_gc"],
        "pval": res.pvalues["batch_imbalance_gc"],
        "r2": res.rsquared,
    })

results = pd.DataFrame(rows)

hits = results.query("r2 > 0.6 and pval < 0.05 and n >= 5")
print(hits.sort_values("r2", ascending=False))

   method_id  n     slope      pval        r2
13      scvi  6 -0.696024  0.002088  0.926311
